In [15]:
import os
import re
from typing import List, Dict, Any

from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import LETTER
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)  # loads variables from .env into environment

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")




In [16]:
load_dotenv()  # loads variables from .env into environment

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

### ISO 8601 to Seconds Converter

In [17]:
#Helper function to convert ISO 8601 duration format to seconds.
ISO8601_RE = re.compile(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?")

def iso8601_duration_to_seconds(d: str) -> int:
    """
    YouTube durations look like 'PT1M30S'. Convert to seconds.
    """
    m = ISO8601_RE.fullmatch(d)
    if not m:
        return 0
    h = int(m.group(1) or 0)
    mins = int(m.group(2) or 0)
    s = int(m.group(3) or 0)
    return h * 3600 + mins * 60 + s




### Tool Youtube Search

In [18]:

# Tool A: Youtube Search
def youtube_search_tool(youtube, query: str, max_results: int = 10) -> List[Dict[str, Any]]:

    # Video Duration is "short" which is a paramter in the search api so that we can filter videos which are less than 2 minutes.
    req = youtube.search().list(
        part="snippet",
        q=query,
        type="video",
        videoDuration="short",
        maxResults=max_results,
    )
    resp = req.execute()
    items = resp.get("items", [])
    out = []
    for it in items:
        vid = it["id"]["videoId"]
        title = it["snippet"]["title"]
        channel = it["snippet"]["channelTitle"]
        out.append({"video_id": vid, "title": title, "channel": channel})
    return out




### Tool Video Details (Duration)

In [19]:
# Tool B: Video Details (to get duration and verify it's short)
def youtube_video_details_tool(youtube, video_ids: List[str]) -> Dict[str, Dict[str, Any]]:
    # contentDetails.duration gives ISO 8601 duration string. :contentReference[oaicite:7]{index=7}
    req = youtube.videos().list(
        part="contentDetails,snippet",
        id=",".join(video_ids)
    )
    resp = req.execute()
    details = {}
    for it in resp.get("items", []):
        vid = it["id"]
        dur = it["contentDetails"]["duration"]
        details[vid] = {
            "duration": dur,
            "duration_seconds": iso8601_duration_to_seconds(dur),
            "title": it["snippet"]["title"],
            "channel": it["snippet"]["channelTitle"],
        }
    return details




### Tool Transcript

In [20]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from youtube_transcript_api._errors import RequestBlocked, IpBlocked
from youtube_transcript_api.proxies import WebshareProxyConfig

PROXY_USER = os.getenv("PROXY_USER")
PROXY_PASS = os.getenv("PROXY_PASS")
PROXY_HOST = os.getenv("PROXY_HOST")
PROXY_PORT = os.getenv("PROXY_PORT")

# Create ONE proxied client globally
ytt_api = YouTubeTranscriptApi(
    proxy_config=WebshareProxyConfig(
        proxy_username=PROXY_USER,
        proxy_password=PROXY_PASS,
    )
)

def transcript_tool(video_id: str, languages=("en",)):
    try:
        fetched = ytt_api.fetch(video_id, languages=list(languages))
        parts = fetched.to_raw_data()
        text = " ".join(p.get("text", "") for p in parts).strip()
        text = " ".join(text.split())

        if not text:
            return {"ok": False, "reason": "EMPTY", "text": ""}

        return {"ok": True, "reason": "OK", "text": text}

    except TranscriptsDisabled:
        return {"ok": False, "reason": "DISABLED", "text": ""}

    except NoTranscriptFound:
        return {"ok": False, "reason": "NOT_FOUND", "text": ""}

    except (RequestBlocked, IpBlocked):
        return {"ok": False, "reason": "IP_BLOCKED", "text": ""}

    except Exception as e:
        return {"ok": False, "reason": f"ERROR:{type(e).__name__}", "text": ""}

### Tool PDF Writer

In [21]:
# ---------- Tool E: PDF writer ----------
def pdf_tool(output_path: str, sections: List[Dict[str, str]]) -> str:
    # Basic ReportLab canvas usage. :contentReference[oaicite:9]{index=9}
    c = canvas.Canvas(output_path, pagesize=LETTER)
    width, height = LETTER
    y = height - 50

    c.setFont("Helvetica-Bold", 14)
    c.drawString(50, y, "YouTube Transcript Summaries")
    y -= 30

    c.setFont("Helvetica", 10)

    for sec in sections:
        lines = [
            f"Title: {sec['title']}",
            f"Channel: {sec['channel']}",
            f"Video ID: {sec['video_id']}",
            "",
            "Summary:",
            sec["summary"],
            "",
            "-" * 90,
            ""
        ]

        for line in lines:
            # simple wrapping
            for chunk in [line[i:i+110] for i in range(0, len(line), 110)]:
                if y < 60:
                    c.showPage()
                    c.setFont("Helvetica", 10)
                    y = height - 50
                c.drawString(50, y, chunk)
                y -= 14

    c.save()
    return output_path




### Defining tools

In [22]:
TOOLS = [
    {
        "type": "function",
        "name": "youtube_search_tool",
        "description": "Search YouTube videos for a query and return candidates with video_id.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "max_results": {"type": "integer", "default": 12}
            },
            "required": ["query"]
        }
    },
    {
        "type": "function",
        "name": "youtube_video_details_tool",
        "description": "Fetch YouTube video details (title, channel, duration_seconds) for video IDs.",
        "parameters": {
            "type": "object",
            "properties": {
                "video_ids": {
                    "type": "array",
                    "items": {"type": "string"}
                }
            },
            "required": ["video_ids"]
        }
    },
    {
        "type": "function",
        "name": "transcript_tool",
        "description": "Fetch transcript for a YouTube video_id.",
        "parameters": {
            "type": "object",
            "properties": {
                "video_id": {"type": "string"}
            },
            "required": ["video_id"]
        }
    },
    {
        "type": "function",
        "name": "pdf_tool",
        "description": "Write a PDF with sections (title/channel/summary) and return the saved path.",
        "parameters": {
            "type": "object",
            "properties": {
                "output_pdf": {"type": "string"},
                "sections": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "video_id":{"type": "string"},
                            "title": {"type": "string"},
                            "channel": {"type": "string"},
                            "summary": {"type": "string"}
                        },
                        "required": ["title", "channel", "summary"]
                    }
                }
            },
            "required": ["output_pdf", "sections"]
        }
    }
]

### The Agent

In [23]:
import json
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Callable


# -----------------------------
# Tools you already have
# -----------------------------
# youtube_search_tool(youtube, query, max_results=12) -> List[Dict]
# youtube_video_details_tool(youtube, ids: List[str]) -> Dict[str, Dict]
# transcript_tool(video_id: str) -> str
# summarize_tool(text: str) -> str
# pdf_tool(output_pdf: str, sections: List[Dict[str, str]]) -> str


# -----------------------------
# Agent state
# -----------------------------
@dataclass
class AgentState:
    goal: str
    max_steps: int = 20
    need: int = 2
    max_seconds: int = 120

    # working memory (your "context" outside the LLM)
    candidates: List[Dict[str, Any]] = field(default_factory=list)
    details: Dict[str, Dict[str, Any]] = field(default_factory=dict)
    chosen: List[Dict[str, Any]] = field(default_factory=list)

    # audit trail (what gets fed back as observations)
    observations: List[str] = field(default_factory=list)


# -----------------------------
# LLM interface (you plug your API here)
# -----------------------------
from openai import OpenAI
from typing import List, Dict, Any
client = OpenAI(api_key=OPENAI_API_KEY)

def call_llm(messages: List[Dict[str, str]]) -> Dict[str, Any]:

    response = client.responses.create(
        model="gpt-4o-mini",
        input=messages,
        tools=TOOLS,
        tool_choice="auto"
    )
    #print(response.output)
    output_type = response.output[0]
    #print(output_type)

    # Case 1: Model wants to call a tool
    if output_type.type == "function_call":
        return {
            "type": "function_call",
            "name": output_type.name,
            "arguments": output_type.arguments,
            "call_id": output_type.call_id
        }

    # Case 2: Model gives final answer
    elif output_type.type == "message":
        return {
            "type": "message",
            "output": output_type.content[0].text
        }

    else:
        #print('expected',output_type.type)
        raise ValueError(f"Unexpected response type: {output_type.type}")




def execute_tool(tool_name: str, args: Dict[str, Any], *, youtube_client: Any, output_pdf: str) -> Any:
    if tool_name == "youtube_search_tool":
        return youtube_search_tool(youtube_client, args["query"], max_results=int(args.get("max_results", 12)))

    if tool_name == "youtube_video_details_tool":
        return youtube_video_details_tool(youtube_client, args["video_ids"])

    if tool_name == "transcript_tool":
        return transcript_tool(args["video_id"])

    if tool_name == "pdf_tool":
        # Ensure output_pdf comes from orchestrator unless LLM explicitly sets it
        pdf_path = args.get("output_pdf", output_pdf)
        return pdf_tool(pdf_path, args["sections"])

    raise ValueError(f"Unknown tool: {tool_name}")




def apply_result_to_state(state: AgentState, tool_name: str, args: Dict[str, Any], result: Any) -> None:
    if tool_name == "youtube_search_tool":
        state.candidates = result

    elif tool_name == "youtube_video_details_tool":
        state.details.update(result)

    elif tool_name == "transcript_tool":
        vid = args["video_id"]
        # store transcript temporarily in details or chosen item later
        state.details.setdefault(vid, {})
        state.details[vid]["transcript"] = result

    elif tool_name == "pdf_tool":
        pass


def summarize_observation_for_llm(state: AgentState, tool_name: str, args: Dict[str, Any], result: Any) -> str:
    """
    Keep observations compact; do NOT dump huge transcripts back (token/cost control).
    Give the model enough to decide next step.
    """
    if tool_name == "youtube_search_tool":
        ids = [x.get("video_id") for x in result][:10]
        return f"youtube_search_tool returned {len(result)} candidates. Example video_ids: {ids}"

    if tool_name == "youtube_video_details_tool":
        # show only the durations/titles for a few
        sample = []
        for vid, det in list(result.items())[:6]:
            sample.append({"video_id": vid, "duration_seconds": det.get("duration_seconds"), "title": det.get("title")})
        return f"youtube_video_details_tool returned details for {len(result)} videos. Sample: {sample}"

    if tool_name == "transcript_tool":
        vid = args["video_id"]
        det = state.details.get(vid, {})
        text = str(result)
        cap = 6000  # keep costs under control; increase if you want
        excerpt = text[:cap]
        return json.dumps({
            "video_id": vid,
            "title": det.get("title", "Unknown"),
            "channel": det.get("channel", "Unknown"),
            "duration_seconds": det.get("duration_seconds", None),
            "transcript_excerpt": excerpt,
            "transcript_truncated": len(text) > cap
        })



    if tool_name == "pdf_tool":
        return f"SUCCESS: pdf_tool wrote PDF at {result}. You must now stop and return final confirmation."

    return f"{tool_name} result type={type(result).__name__}"


def ensure_pdf_written(state: AgentState, youtube_client: Any, output_pdf: str) -> str:
    sections = []
    for x in state.chosen:
        sections.append(
            {
                "video_id": x["video_id"],
                "title": x["title"],
                "channel": x["channel"],
                "summary": x["summary"],
            }
        )
    return pdf_tool(output_pdf, sections)


def state_max_seconds_str(max_seconds: int) -> str:
    return f"{max_seconds} seconds"

def append_tool_output(messages: List[Dict[str, Any]], call_id: str, output: Any) -> None:
    # Responses API expects tool outputs as input items, not role="tool" messages
    messages.append({
        "type": "function_call_output",
        "call_id": call_id,
        "output": output if isinstance(output, str) else json.dumps(output)
    })


# -----------------------------
# Recommended improvement:
# Make summarize_tool accept video_id too
# -----------------------------

In [24]:
# -----------------------------
# Agentic ReAct-style loop

# -----------------------------
from googleapiclient.discovery import build

def run_agent_agentic(query: str, output_pdf: str = "youtube_summaries.pdf") -> str:

    api_key = YOUTUBE_API_KEY
    if not api_key:
        raise RuntimeError("Set env var YOUTUBE_API_KEY with your YouTube Data API key.")
    youtube = build("youtube", "v3", developerKey=api_key)

    state = AgentState(goal=f"Find <= {state_max_seconds_str(120)} YouTube videos about '{query}', get top {2} transcripts, summarize, save to PDF.")  # we’ll override below
    state.goal = (
        f"User goal: Find YouTube videos about '{query}' that are <= {state.max_seconds}s, "
        f"collect transcripts for {state.need} videos, summarize each, and save a PDF to '{output_pdf}'."
    )
    sys_prompt = f"""
    You are a tool-using agent. Your job is to produce a PDF that contains summaries for exactly {state.need} YouTube videos about the user query.

    Hard requirements (must always hold):
    - You must end with exactly {state.need} chosen videos.
    - Only choose videos where duration_seconds <= {state.max_seconds}.
    - Only choose videos where a transcript is available (transcript_tool returns a non-empty string).
    - The final step MUST be calling pdf_tool successfully. Do NOT output a final message before pdf_tool succeeds.

    Process you should follow:
    1) Find candidate videos and their details, then filter by duration and transcript availability.
    2) Select candidates that meet the duration constraint, then collect transcripts  on them until you have {state.need} non-empty transcripts.
    3) Once you have {state.need} transcripts, prepare short summaries for each transcript in your response text (the orchestrator will store them). Keep each summary concise and useful.
    4) make a pdf with sections for the {state.need} chosen videos (video_id, title, channel, summary). This is the last tool call.

    Tool-use rules:
    - Use tools only using function calls (no plain text tool requests).
    - If a transcript is missing/empty, skip that video_id and try another.
    - If you cannot find enough valid videos, keep trying more candidates from the search results. Only give up if you have exhausted all candidates.

    Output format rules (strict):
    - If you need to use a tool, output ONLY a JSON object in exactly this shape:
    {{ "type": "function_call", "name": "<tool_name>", "arguments": {{ ... }} }}
    - If you are done (only after pdf_tool succeeded), output ONLY:
    {{ "type": "message", "output": "<short confirmation including the pdf path>" }}

    Available tools:
    {json.dumps(TOOLS, indent=2)}
    """.strip()
    

    messages: List[Dict[str, str]] = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": state.goal},
    ]
    

    for step in range(state.max_steps):
        # Ask LLM: what next?
        llm_reply = call_llm(messages)

        if llm_reply.get("type") == "message":
            # LLM says done (but we still ensure a PDF exists)
            if not state.chosen or len(state.chosen) < state.need:
                raise RuntimeError(f"LLM ended early: only have {len(state.chosen)}/{state.need} videos.")
            # If pdf not created yet, create it here as a safety net
            return ensure_pdf_written(state, youtube, output_pdf)

        if llm_reply.get("type") != "function_call":
            raise RuntimeError(f"Invalid LLM reply: {llm_reply}")

        tool_name = llm_reply["name"]
        args = llm_reply.get("arguments", "{}")
        args = json.loads(args) if isinstance(args, str) else args
        call_id = llm_reply["call_id"]
        print(llm_reply)

        # Execute tool
        try:
            result = execute_tool(tool_name, args, youtube_client=youtube, output_pdf=output_pdf)
        except Exception as e:
            obs = f"TOOL_ERROR {tool_name}: {type(e).__name__}: {e}"
            state.observations.append(obs)
            # Add the actual function_call item to the input history
            messages.append({
                "type": "function_call",
                "call_id": call_id,
                "name": llm_reply["name"],
                "arguments": llm_reply["arguments"]  # must be a JSON string here (yours already is)
            })

            # Add the matching output item
            messages.append({
                "type": "function_call_output",
                "call_id": call_id,
                "output": obs
            })
            continue

        # Update our explicit state (optional but recommended)
        apply_result_to_state(state, tool_name, args, result)

        # Feed observation back
        obs = summarize_observation_for_llm(state, tool_name, args, result)
        state.observations.append(obs)

        

        # Add the actual function_call item to the input history
        messages.append({
            "type": "function_call",
            "call_id": call_id,
            "name": llm_reply["name"],
            "arguments": llm_reply["arguments"]  # must be a JSON string here (yours already is)
        })

        # Add the matching output item
        messages.append({
            "type": "function_call_output",
            "call_id": call_id,
            "output": obs
        })

        # Hard stop if we already wrote PDF (agent can still “final” next)
        if tool_name == "pdf_tool":
            # result is expected to be pdf path
            messages.append({"type": "message", "call_id": call_id,"output": f"Saved PDF: {result}"})
            #messages.append({"role": "assistant", "content": json.dumps({"type": "final", "output": f"Saved PDF: {result}"})})
            return str(result)

    raise RuntimeError(f"Max steps exceeded ({state.max_steps}). Observations:\n" + "\n".join(state.observations))

In [ ]:
pdf_path = run_agent_agentic("python datascience with transcripts", output_pdf="movies_summaries.pdf")
print("PDF saved at:", pdf_path)